In [1]:
def build_dp(reference_tokens, hypothesis_tokens):
    """
    计算 Reference -> Hypothesis 的最小编辑距离 DP 表。

    参数：
        reference_tokens: 参考 token 列表
        hypothesis_tokens: 预测 token 列表

    返回：
        dp: 二维动态规划表
    """

    m = len(reference_tokens)
    n = len(hypothesis_tokens)

    #1.创建DP表
    dp = [
        [0] * (n+1)
        for _ in range(m+1)
    ]

    #2.初始化：Reference -> 空序列
    for i in range(m+1):
        dp[i][0] = i

    #3. 初始化：空序列 -> Hypothesis
    for j in range(n+1):
        dp[0][j] = j

    #4. 状态转移
    for i in range(1, m+1):
        for j in range(1, n+1):

            cost = (
                0
                if reference_tokens[i-1] == hypothesis_tokens[j-1]
                else 1
            )

            deletion = dp[i-1][j] + 1
            insertion = dp[i][j-1] + 1
            substitution = dp[i-1][j-1] + cost

            dp[i][j] = min(deletion,insertion,substitution)

    return dp

#测试
reference = "I like apples".split()
hypothesis = "I love apples today".split()

dp = build_dp(reference,hypothesis)

for row in dp:
    print(row)

print("\n编辑距离:",dp[-1][-1])
print("WER:", dp[-1][-1] / len(reference))

[0, 1, 2, 3, 4]
[1, 0, 1, 2, 3]
[2, 1, 1, 2, 3]
[3, 2, 2, 1, 2]

编辑距离: 2
WER: 0.6666666666666666


In [7]:
#运行试验
import sys
from pathlib import Path

root = Path(r"D:\Projects\speech-ai-learning")

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.metrics import score_tokens

reference = "I like apples".split()
hypothesis = "I love apples today".split()

result = score_tokens(reference,hypothesis)

print("编辑操作：")

for op, ref_token, hyp_token in result["operations"]:
    print(f"{op}: {ref_token} -> {hyp_token}")

print("\n错误统计：")

for key in ["S", "D", "I", "C", "N", "distance", "error_rate"]:
    print(f"{key}: {result[key]}")

assert result["S"] == 1
assert result["D"] == 0
assert result["I"] == 1
assert result["C"] == 2
assert result["distance"] == 2

print("\nBacktrace验收通过")

编辑操作：
C: I -> I
S: like -> love
C: apples -> apples
I: None -> today

错误统计：
S: 1
D: 0
I: 1
C: 2
N: 3
distance: 2
error_rate: 0.6666666666666666

Backtrace验收通过


In [3]:
#jiwer计算与自己实现对照
from jiwer import process_words
from src.metrics import score_tokens

reference = "I like apples"
hypothesis = "I love apples today"

#自己实现的算法
ours = score_tokens(
    reference.split(),
    hypothesis.split()
)

#jiwer
reference_result = process_words(
    reference,
    hypothesis
)

print("========== Ours ==========")
print(ours)

print("\n========== jiwer ==========")
print("S:",result.substitutions)
print("D:", result.deletions)
print("I:", result.insertions)
print("C:", result.hits)
print("WER:", result.wer)

# 核对
assert ours["S"] == reference_result.substitutions
assert ours["D"] == reference_result.deletions
assert ours["I"] == reference_result.insertions
assert ours["C"] == reference_result.hits

assert abs(
    ours["error_rate"] - reference_result.wer
) < 1e-8

print("\njiwer 对照通过")

========== Ours ==========
{'S': 1, 'D': 0, 'I': 1, 'C': 2, 'N': 3, 'distance': 2, 'error_rate': 0.6666666666666666, 'operations': [('C', 'I', 'I'), ('S', 'like', 'love'), ('C', 'apples', 'apples'), ('I', None, 'today')]}

========== jiwer ==========
S: 1
D: 0
I: 1
C: 2
WER: 0.6666666666666666

jiwer 对照通过


In [4]:
#空词与重复词测试
from src.metrics import score_tokens
from jiwer import process_words

test_cases = [
    #两者都为空
    ([],[]),

    #Reference为空
    ([],["I", "like", "apples"]),

    # Hypothesis 为空
    (["I", "like", "apples"], []),

    # 重复词
    (["a", "a", "b"], ["a", "b"]),
]

for reference, hypothesis in test_cases:
    result = score_tokens(reference, hypothesis)

    print("=" * 40)
    print("Reference:", reference)
    print("Hypothesis:", hypothesis)

    print(
        f"S={result['S']}, "
        f"D={result['D']}, "
        f"I={result['I']}, "
        f"C={result['C']}"
    )

    print("Distance:", result["distance"])
    print("WER:", result["error_rate"])

    assert (
        result["S"] + result["D"] + result["I"]
        == result["distance"]
    )

    assert (
        result["C"] + result["S"] + result["D"]
        == len(reference)
    )

print("\n边界测试全部通过")

Reference: []
Hypothesis: []
S=0, D=0, I=0, C=0
Distance: 0
WER: None
Reference: []
Hypothesis: ['I', 'like', 'apples']
S=0, D=0, I=3, C=0
Distance: 3
WER: None
Reference: ['I', 'like', 'apples']
Hypothesis: []
S=0, D=3, I=0, C=0
Distance: 3
WER: 1.0
Reference: ['a', 'a', 'b']
Hypothesis: ['a', 'b']
S=0, D=1, I=0, C=2
Distance: 1
WER: 0.3333333333333333

边界测试全部通过


In [5]:
#jiwer对空字符串处理
for reference, hypothesis in [
    ("", ""),
    ("", "I like apples"),
    ("I like apples", ""),
]:
    result = process_words(
        reference,
        hypothesis
    )

    print("=" * 40)
    print("Reference:", repr(reference))
    print("Hypothesis:", repr(hypothesis))

    print("S:", result.substitutions)
    print("D:", result.deletions)
    print("I:", result.insertions)
    print("C:", result.hits)
    print("WER:", result.wer)

Reference: ''
Hypothesis: ''
S: 0
D: 0
I: 0
C: 0
WER: 0
Reference: ''
Hypothesis: 'I like apples'
S: 0
D: 0
I: 3
C: 0
WER: 3
Reference: 'I like apples'
Hypothesis: ''
S: 0
D: 3
I: 0
C: 0
WER: 1.0


In [15]:
#证明两种平均方式不同
from jiwer import process_words
import sys
from pathlib import Path

root = Path(r"D:\Projects\speech-ai-learning")

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.metrics import score_tokens,score_corpus

references = [
    "hello",
    "one two three four five six seven eight nine",
]

hypotheses = [
    "",
    "one two three four five six seven eight nine",
]

token_pairs = [
    (ref.split(), hyp.split())
    for ref, hyp in zip(references, hypotheses)
]

# 手写语料级指标
ours = score_corpus(token_pairs)

# jiwer语料级对照
jw = process_words(references, hypotheses)

# 错误方式：平均单句WER
sentence_mean = sum(
    r["error_rate"]
    for r in ours["sample_results"]
) / len(ours["sample_results"])

print("总参考词数:", ours["N"])
print("总编辑次数:", ours["distance"])
print("直接平均单句WER:", sentence_mean)
print("正确语料级WER:", ours["error_rate"])
print("jiwer语料级WER:", jw.wer)

assert ours["N"] == 10
assert ours["distance"] == 1
assert abs(ours["error_rate"] - 0.1) < 1e-8
assert abs(ours["error_rate"] - jw.wer) < 1e-8

print("\n语料级WER验收通过")

总参考词数: 10
总编辑次数: 1
直接平均单句WER: 0.5
正确语料级WER: 0.1
jiwer语料级WER: 0.1

语料级WER验收通过


In [16]:
#验证空reference样本
pairs = [
    ([], ["extra"]),
    (["hello"], ["hello"]),
]

result = score_corpus(pairs)

print("总参考词数:", result["N"])
print("总插入次数:", result["I"])
print("语料级错误率:", result["error_rate"])

assert result["N"] == 1
assert result["I"] == 1
assert result["error_rate"] == 1.0

总参考词数: 1
总插入次数: 1
语料级错误率: 1.0
